In [ ]:
import pandas as pd
import openpyxl
import re
import os
import sys

EXCEL = r"Survey.xlsx"

def rename_columns(excel=EXCEL):

    df = pd.read_excel(excel)


    Old_Columns = list(df.columns)

    New_Columns = []

    for old_column in Old_Columns:
        new_column = old_column.split("-")[-1]
        new_column = new_column.strip()
        df = df.rename(columns={
            old_column: new_column
        })
    

    return df

renamed_columns_df = rename_columns()

def clean_role(val):
    """Q1 Column"""
    if pd.isna(val):
        return val

    val = str(val).strip()
    val = val.split(":")[-1].strip()
    return val.lower()

renamed_columns_df["Which Title Best Fits your Current Role?"] = renamed_columns_df["Which Title Best Fits your Current Role?"].apply(clean_role)

role_map = {
    r'data\s*engineer':                 'Data Engineer',
    r'software\s*(developer|engineer)': 'Software Developer',
    r'\bai\b|machine learning|\bml\b':  'AI/ML Engineer',
    r'data\s*scien':                    'Data Scientist',
    r'data\s*analy':                    'Data Analyst',
    r'business\s*analy':                'Business Analyst',
    r'financ(e|al)':                    'Finance',
    r'manufact':                        'Manufacturing',
    r'non\s*profit':                    'Nonprofit',
    r'consulti':                        'Consulting'


}

def role_standardize(val):
    if pd.isna(val) or val == "":
        return val
    for pattern, label in role_map.items():
        if re.search(pattern, val):
            return label
    return val.title()

renamed_columns_df["Which Title Best Fits your Current Role?"]  = renamed_columns_df["Which Title Best Fits your Current Role?"].apply(role_standardize)

def clean_industry(val):

    """Q4. Column contains basic entries for industries as well as open answers. 
    The format is - 'Other (Please Specify): <industry>. 
    This function cleans that to return only the industry of each cell. 
    The function is applied to the column using pandas.Series.apply()
    Args:
        val (str): An entry in the column
    Returns:
        str: The formatted entry
        
        """
        
    if pd.isna(val):
        return val
    val = str(val).strip()
    val = val.split(":")[-1].strip()
    return val.lower()


renamed_columns_df["What Industry do you work in?"] = renamed_columns_df["What Industry do you work in?"].apply(clean_industry)

industry_map ={
    r'financ(e|al)':                    'Finance',
    r'manufact':                        'Manufacturing',
    r'non\s*profit':                    'Nonprofit',
    r'consulti':                        'Consulting'

}

def industry_standardize(val):
    if pd.isna(val) or val == "":
        return val
    for pattern, label in industry_map.items():
        if re.search(pattern, val):
            return label
    return val.title()

renamed_columns_df["What Industry do you work in?"] = renamed_columns_df["What Industry do you work in?"].apply(industry_standardize)


def clean_country(val):
    if pd.isna(val):
        return val

    val = str(val).strip()
    val = val.split(":")[-1]
    return val.lower()

renamed_columns_df["Which Country do you live in?"] = renamed_columns_df["Which Country do you live in?"].apply(clean_country)

country_map = {
    r'nigeri':              'Nigeria',
    r'argentin':            'Argentina',
    r'brazi':               'Brazil',
    r'^fin':                'Finland', 
    r'^(ire|irel)$':        'Ireland',
    r'^leba':               'Lebanon',
    r'perú':                'Peru',
    r'portug':              'Portugal',
    r'^uae$':               'United Arab Emirates',
    r'^uzb':                'Uzbekistan',
}

def standardize_country(val):
    if pd.isna(val) or val == "":
        return val
    for pattern, label in country_map.items():
        if re.search(pattern, val):
            return label
    return val.title()



renamed_columns_df["Which Country do you live in?"] = renamed_columns_df["Which Country do you live in?"].apply(standardize_country)
# list(renamed_columns_df["Which Country do you live in?"])



def clean_language(val):
    if pd.isna(val):
        return val
    val = str(val).strip()


    # ---- evaluate for SQL first ----#
    if "sql" in val.lower():
        return "SQL"

    val = val.split(":")[0].strip().title()

    return val.title()


renamed_columns_df["Favorite Programming Language"] = renamed_columns_df["Favorite Programming Language"].apply(clean_language)


def clean_important(val):
    if pd.isna(val):
        return val
    val = str(val).strip()
    if val.startswith("Other"):
        return "Other"
    return val
    
renamed_columns_df["If you were to look for a new job today, what would be the most important thing to you?"] = renamed_columns_df["If you were to look for a new job today, what would be the most important thing to you?"].apply(clean_important)


def clean_ethnicity(val):
    if pd.isna(val):
        return val
    val = str(val).strip()
    if val.startswith("Other"):
        return "Other"
    return val


renamed_columns_df["Ethnicity"] = renamed_columns_df["Ethnicity"].apply(clean_ethnicity)

df = renamed_columns_df.drop(renamed_columns_df.columns[3:10], axis=1)
df.to_excel('Survey_Clean.xlsx', index=False)


# Columns to drop: Time Taken (America/New_York),	Browser,	OS,	City,	Country,	Referrer,	Time Spent

